# W&B Models Introduction Hands-on

このノートブックでは、W&B Modelsの主要機能を小さなサンプルで順番に試します。

1. Experiment Tracking
2. Tables and Rich Media
3. Artifacts
4. Sweeps
5. Registry
6. Reports
7. Extended Capabilities

各章は同じフォルダにあるPythonスクリプトを実行します。スクリプトとノートブックで
Artifact名やメトリクス名がずれないため、1章から順番に進めてください。


## Setup

この教材は `models_handson` ディレクトリを uv プロジェクトとして管理します。
ターミナルで次を実行してください。ロックファイルでは W&B SDK 0.28.1 を使用します。

```bash
cd models_handson
uv sync
uv run jupyter notebook
```

環境変数は実行前に設定してください。

```bash
export WANDB_ENTITY="your-team-entity"
export WANDB_PROJECT="wandb-models-handson"  # 省略可能
# Dedicated Cloud / Self-Managedの場合のみ
export WANDB_BASE_URL="https://your-wandb-host"
```

API Keyは `WANDB_API_KEY` に設定するか、`uv run wandb login` で保存します。ソースコードへ
API Key、Team名、Project名を書き込まないでください。


In [ ]:
# パッケージはターミナルから準備します。
# cd models_handson
# uv sync
# uv run jupyter notebook


In [ ]:
import os
from pathlib import Path
import runpy
import subprocess
import sys

import wandb

if not os.getenv("WANDB_ENTITY"):
    raise RuntimeError("WANDB_ENTITY にW&BのTeam Entityを設定してください。")

os.environ.setdefault("WANDB_PROJECT", "wandb-models-handson")
wandb.login()

# Notebookをjp内、models_handson直下、リポジトリ直下のどこから開いても動作します。
candidates = [
    Path.cwd(),
    Path.cwd() / "jp",
    Path.cwd() / "models_handson" / "jp",
]
HANDSON_DIR = next(
    (candidate for candidate in candidates if (candidate / "1_experiment.py").exists()),
    None,
)
if HANDSON_DIR is None:
    raise RuntimeError("models_handson/jpディレクトリを作業ディレクトリから見つけられません。")
HANDSON_DIR = HANDSON_DIR.resolve()

print(f"Entity: {os.environ['WANDB_ENTITY']}")
print(f"Project: {os.environ['WANDB_PROJECT']}")
print(f"Scripts: {HANDSON_DIR}")


# 1. Experiment Tracking

Config、Tags、Group、Job TypeでRunを整理し、階層化したメトリクスとカスタムX軸を
記録します。5つの模擬実験は固定Seedを使うため、同じ条件なら同じ値になります。

実行後はWorkspaceで次を確認します。

- `train/loss`、`val/loss`、`val/accuracy` の比較
- `models-handson-baselines` Group
- `handson`、`experiment`、`baseline` Tags
- `custom/epoch_squared` をX軸にした `custom/inverse_epoch`


In [ ]:
runpy.run_path(str(HANDSON_DIR / "1_experiment.py"), run_name="__main__")


# 2. Tables and Rich Media

外部データの大容量ダウンロードは行わず、NumPyで生成した画像とセグメンテーション
マスクを使用します。単一画像、Mask Overlay、Table、StepごとのTableを記録します。

実行後はWorkspaceで `examples/table` と `examples/progression` を開き、行の比較と
Step切り替えを試してください。


In [ ]:
runpy.run_path(str(HANDSON_DIR / "2_table.py"), run_name="__main__")


# 3. Artifacts

ArtifactはデータやモデルをVersion管理する仕組みです。この章では、データ生成、
前処理、模擬学習を別Runとして実行し、次のArtifactを作ります。

- `models-handson-dataset`
- `models-handson-processed-dataset`
- `models-handson-model`

`use_artifact()` で入力を宣言するため、ArtifactsのLineageに
dataset → processed dataset → model の関係が表示されます。


In [ ]:
runpy.run_path(str(HANDSON_DIR / "3_artifacts.py"), run_name="__main__")


# 4. Sweeps

Random Sweepで `learning_rate`、`epochs`、`batch_size` を探索します。最適化対象は
`val/accuracy` です。ローカルAgentは6試行で終了します。

実行後はSweep画面で、Parallel Coordinates、Parameter Importance、各Trialを確認します。


In [ ]:
# W&B ServerへSweepを作り、ローカルで6 Trialを実行します。
runpy.run_path(str(HANDSON_DIR / "4_sweeps.py"), run_name="__main__")


# 5. Registry

Registryは、Projectで作成したArtifact Versionを組織全体で管理する中央リポジトリです。
ArtifactをCollectionへリンクしても、実体ファイルは複製されません。

## GUIでの手順

1. [W&B Registry](https://wandb.ai/registry/)を開きます。
2. 既存Registryを選ぶか、**Create registry**で名前、Visibility、Artifact typeを設定します。
3. **Create collection**でCollection名と受け入れるArtifact typeを設定します。
4. **Link version**からProject、Artifact、Versionを選択します。
5. Alias、Tags、Collection card、Registry accessを必要に応じて設定します。
6. Versionの**Lineage**とCollectionの**Action History**を確認します。

## Programmaticな手順

次のセルは3章の `models-handson-model:latest` をRegistryへリンクし、`candidate` Alias、
Tags、Collection cardを設定した後、Registry経由で再取得します。

既定値:

- `WANDB_REGISTRY=Models`
- `WANDB_REGISTRY_COLLECTION=models-handson-model`

別のRegistryを使う場合はセル実行前に環境変数を変更してください。新規Registryの作成には
適切な権限が必要です。


In [ ]:
os.environ.setdefault("WANDB_REGISTRY", "Models")
os.environ.setdefault("WANDB_REGISTRY_COLLECTION", "models-handson-model")
runpy.run_path(str(HANDSON_DIR / "5_registry.py"), run_name="__main__")


## Registry 参考ドキュメント

- [Registry overview](https://docs.wandb.ai/models/registry)
- [Create a registry](https://docs.wandb.ai/models/registry/create_registry)
- [Configure registry access](https://docs.wandb.ai/models/registry/configure_registry)
- [Create a collection](https://docs.wandb.ai/models/registry/create_collection)
- [Link an artifact version](https://docs.wandb.ai/models/registry/link_version)
- [Link a Weave prompt](https://docs.wandb.ai/models/registry/link_prompt)
- [Aliases](https://docs.wandb.ai/models/registry/aliases)
- [Download an artifact](https://docs.wandb.ai/models/registry/download_use_artifact)
- [Find registry items](https://docs.wandb.ai/models/registry/search_registry)
- [Organize with tags](https://docs.wandb.ai/models/registry/organize-with-tags)
- [Collection cards](https://docs.wandb.ai/models/registry/registry_cards)
- [Lineage and audit history](https://docs.wandb.ai/models/registry/lineage)
- [Delete registry](https://docs.wandb.ai/models/registry/delete_registry)


# 6. Reports

ReportsはRunの可視化と考察を一つにまとめ、共同編集・共有する機能です。

## GUIでの手順

1. ProjectのWorkspaceを開き、右上の**Create report**を選びます。
2. 開始時に含めるPanelを選びます。後から追加・削除できます。
3. 本文、見出し、Panel Grid、Run set、グラフを編集します。
4. **Publish to project**で公開し、**Share**から共有します。

## 主な選択肢

- 文章: Heading、Markdown、List、Code、Image、Media、Table of contents
- Run整理: Runset、PanelGrid、Filter、Group、RunComparer、CodeComparer
- 基本グラフ: LinePlot、BarPlot、ScatterPlot、ScalarChart、CustomChart
- Sweep分析: ParallelCoordinatesPlot、ParameterImportancePlot
- データ/Artifact: Summary Table、Artifact、Artifact File Panel

## Programmaticな手順

Report and Workspace APIはPublic Previewです。この例ではLinePlotと
ParameterImportancePlotの2種類だけを使い、Draftとして保存します。


In [ ]:
runpy.run_path(str(HANDSON_DIR / "6_report.py"), run_name="__main__")


## Reports 参考リンク

公式ドキュメント:

- [Reports overview](https://docs.wandb.ai/models/reports)
- [Create a report](https://docs.wandb.ai/models/reports/create-a-report)
- [Edit a report](https://docs.wandb.ai/models/reports/edit-a-report)
- [Collaborate](https://docs.wandb.ai/models/reports/collaborate-on-reports)
- [Clone and export](https://docs.wandb.ai/models/reports/clone-and-export-reports)
- [Embed](https://docs.wandb.ai/models/reports/embed-reports)
- [Cross-project reports](https://docs.wandb.ai/models/reports/cross-project-reports)
- [Reports gallery](https://docs.wandb.ai/models/reports/reports-gallery)
- [Report API Quickstart](https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Report_API_Quickstart.ipynb)

サンプルReport:

- [Nejumi LLMリーダーボード4](https://wandb.ai/llm-leaderboard/nejumi-leaderboard4/reports/Nejumi-LLM-4--VmlldzoxMzc1OTk1MA)
- [OpenPI and W&B for Physical AI (EN)](https://wandb.ai/wandb-smle/openpi-aloha-wandb-integration/reports/OpenPI-and-W-B-for-Physical-AI-Experiment-tracking-guide--VmlldzoxNjQyMzc4Mg)
- [OpenPI x W&B (JP)](https://wandb.ai/wandb-smle/openpi-aloha-wandb-integration/reports/OpenPI-x-W-B--VmlldzoxNjQxNjczNQ)


# 7. Extended Capabilities

追加機能はCLIの `--demo` で一つずつ試せます。

- `resume`: 同じRun IDへ追記
- `query`: Public APIで必要なメトリクスだけを取得
- `charts`: PR CurveとAudio
- `offline`: Offline Runと後からのsync
- `alert`: 外部通知。`--enable-alert` がある場合だけ送信
- `settings`: よく使う環境変数とSettings

下のセルは既定のResumeデモだけを実行します。Alertは自動実行しません。


In [ ]:
subprocess.run(
    [sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"), "--demo", "resume"],
    check=True,
)


他のデモを試す場合:

```python
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "charts"], check=True)
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "offline"], check=True)
```

Alertは実際に外部通知を送るため、意図した場合だけ次を実行してください。

```python
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "alert", "--enable-alert"], check=True)
```


# まとめ

- RunへConfig、メトリクス、Table、Mediaを記録する
- ArtifactsでデータとモデルをVersion管理し、Lineageを残す
- Sweepsで探索を自動化する
- Registryで組織向けのArtifact Versionをキュレーションする
- Reportsで結果と考察を共有する

すべての章は同じ `WANDB_ENTITY/WANDB_PROJECT` を使用します。Project Workspace、
Artifacts、Sweeps、Registry、Reportsを行き来して、各オブジェクトのつながりを確認してください。
